In [1]:
"""
# PhishGuard AI - Train and Compare Model

## Objective

This notebook trains and compares two machine learning classifiers:

1. Decision Tree Classifier
2. Random Forest Classifier

The models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

The model with the highest phishing-class F1 score will be selected as the deployment model.

## Label Convention

- 0 = Legitimate
- 1 = Phishing
"""

'\n# PhishGuard AI - Train and Compare Model\n\n## Objective\n\nThis notebook trains and compares two machine learning classifiers:\n\n1. Decision Tree Classifier\n2. Random Forest Classifier\n\nThe models are evaluated using:\n\n- Accuracy\n- Precision\n- Recall\n- F1-score\n\nThe model with the highest phishing-class F1 score will be selected as the deployment model.\n\n## Label Convention\n\n- 0 = Legitimate\n- 1 = Phishing\n'

In [2]:

import json
import os
import string
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier



In [3]:

sys.path.insert(0, os.path.abspath("../backend"))
from app.features import (  # noqa: E402
    URL_ONLY_FEATURES,
    canonicalize_url,
    compute_features,
)


In [4]:
RANDOM_STATE = 42
CSV = "../PhiUSIIL_Phishing_URL_Dataset.csv"
OUT = "../outputs"
os.makedirs(OUT, exist_ok=True)

In [5]:
if not os.path.exists(CSV):
    raise SystemExit(f"Dataset not found: {CSV} (place PhiUSIIL CSV in repo root)")

df = pd.read_csv(CSV, encoding="utf-8").drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=["URL"]).reset_index(drop=True)
print(f"Loaded {len(df):,} unique rows from {CSV}")

# Targeted legitimate augmentation: PhiUSIIL's "legitimate" class is 100%
# bare homepage URLs (no path, no query string, no digits, always https --
# see manual_test_results.csv). Add real legitimate URLs that DO have a
# path / query string / plain http so the model sees that those are
# normal, not just a phishing tell.
AUG_CSV = "../legit_augmentation.csv"
df["_is_aug"] = 0
if os.path.exists(AUG_CSV):
    aug = pd.read_csv(AUG_CSV, encoding="utf-8")
    before = len(df)
    df = df[["URL", "label"]].copy()
    df["_is_aug"] = 0
    aug = aug[["URL", "label"]].copy()
    aug["_is_aug"] = 1
    df = pd.concat([df, aug], ignore_index=True)
    df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)
    print(f"Augmented with {len(aug):,} curated legitimate URLs "
          f"({before:,} -> {len(df):,} total rows)")

# ---------------------------------------------------------------------------
# Fix 1 - realistic legitimate URLs. legit_augmentation.csv is small and
# hand-made; legit_deeplinks_sitemaps.csv holds REAL page URLs harvested
# from the public sitemaps of Tranco-ranked websites (see
# harvest_legit_deeplinks.py). Domains used by the evaluation scripts are
# excluded at harvest time.
#
# Split: PhiUSIIL (+ curated aug) rows are split stratified 68/12/20
# (train/validation/test). Harvested URLs are split BY DOMAIN 70/15/15, so
# validation/test pages come from websites the model never saw in training.
# The validation split is only used to tune the decision thresholds (fix 5).
# ---------------------------------------------------------------------------
df["_src"] = np.where(df["_is_aug"] == 1, "aug", "phiusiil")
idx_trainval, idx_test = train_test_split(
    df.index, test_size=0.20, random_state=RANDOM_STATE, stratify=df["label"],
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.15, random_state=RANDOM_STATE,
    stratify=df.loc[idx_trainval, "label"],
)
df["_split"] = "train"
df.loc[idx_val, "_split"] = "val"
df.loc[idx_test, "_split"] = "test"

HARVEST_CSV = "../legit_deeplinks_sitemaps.csv"
harv = pd.read_csv(HARVEST_CSV, encoding="utf-8").drop_duplicates(subset=["URL"])
rng = np.random.default_rng(RANDOM_STATE)
domains = sorted(harv["domain"].unique())
domains = [domains[i] for i in rng.permutation(len(domains))]
n_dom = len(domains)
dom_split = {
    d: "train" if i < 0.70 * n_dom else ("val" if i < 0.85 * n_dom else "test")
    for i, d in enumerate(domains)
}
harv["_split"] = harv["domain"].map(dom_split)

# Realistic variants of real pages: plain http:// (many legitimate sites
# are still reachable over http) and ordinary non-tracking query strings
# (pagination, search, ids, language). Generated per split, so no domain
# crosses splits.
_QUERY_TEMPLATES = [
    "page={n}", "p={n}", "id={id}", "lang=en", "sort=newest&page={n}",
    "q={word}", "s={word}", "category={word}", "ref_id={id}", "v={n}",
]


def _variants(url, r):
    out = []
    if r.random() < 0.20 and url.startswith("https://"):
        out.append("http://" + url[len("https://"):])
    if r.random() < 0.30 and "?" not in url:
        words = [w for w in url.split("/")[3:] if w.isalpha() and 2 < len(w) < 15]
        t = _QUERY_TEMPLATES[r.integers(len(_QUERY_TEMPLATES))]
        q = t.format(n=int(r.integers(1, 40)), id=int(r.integers(100, 999999)),
                     word=(words[0] if words else "news"))
        out.append(url.rstrip("/") + "/?" + q if url.endswith("/") else url + "?" + q)
    return out


extra = [
    {"URL": v, "label": 1, "domain": dom, "_split": part}
    for url, dom, part in harv[["URL", "domain", "_split"]].values
    for v in _variants(url, rng)
]
harv = pd.concat([harv, pd.DataFrame(extra)], ignore_index=True)
harv["_is_aug"] = 0
harv["_src"] = "harvest"
print(f"Harvested deep links: {len(harv):,} URLs (incl. {len(extra):,} http/query "
      f"variants) from {n_dom:,} domains -> "
      + ", ".join(f"{k}={v:,}" for k, v in harv["_split"].value_counts().items()))

df = pd.concat([df, harv[["URL", "label", "_is_aug", "_src", "_split"]]], ignore_index=True)
df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)

# ---------------------------------------------------------------------------
# Recent phishing from threat feeds, with a TIME-BASED split
# (phish_feed_urls.csv from harvest_phish_feeds.py; url, first_seen, source).
# PhiUSIIL phishing is 2019-2023, so on current OpenPhish URLs the model
# gave ~18% a "legitimate" verdict. Feed URLs are grouped by registered
# domain (or full host on shared hosting platforms, e.g. x.mybluehost.me)
# so a whole campaign sits in exactly one split:
#   future : URLs first seen in 2026 (phishunt / OpenPhish) - never trained
#            on; reported separately as the "future feed" detection rate
#   val    : groups first seen Nov-Dec 2025 (used to tune FEED_WEIGHT and
#            the thresholds)
#   train  : groups first seen before Nov 2025
# Groups that also appear in the future rows or in realworld_eval_set.csv
# are dropped from train/val, so no test campaign leaks into training.
# ---------------------------------------------------------------------------
from app.reputation import _extract_icann, _host, is_user_content_host  # noqa: E402

FEED_CSV = "../phish_feed_urls.csv"
FEED_VAL_FROM, FEED_FUTURE_FROM = "2025-11-01", "2026-01-01"
FEED_PER_GROUP = 2          # max URLs per domain/host group (limits campaign bias)
FEED_TRAIN_N, FEED_VAL_N = 60_000, 10_000


def feed_group(u):
    h = _host(u)
    if not h:
        return ""
    return h if is_user_content_host(u) else (_extract_icann(h).top_domain_under_public_suffix or h)


feed = pd.read_csv(FEED_CSV, encoding="utf-8")
feed["g"] = feed["url"].map(feed_group)
is_future = feed["first_seen"] >= FEED_FUTURE_FROM
rw = pd.read_csv("../realworld_eval_set.csv", encoding="utf-8")
held_out = set(feed.loc[is_future, "g"]) | set(rw["url"].map(feed_group))
past = feed[~is_future & ~feed["g"].isin(held_out)].copy()
g_first = past.groupby("g")["first_seen"].min()
past["_split"] = np.where(past["g"].map(g_first) >= FEED_VAL_FROM, "val", "train")
past = past.sample(frac=1, random_state=RANDOM_STATE).groupby("g").head(FEED_PER_GROUP)
past = pd.concat([
    past[past["_split"] == "train"].head(FEED_TRAIN_N),
    past[past["_split"] == "val"].head(FEED_VAL_N),
])
future = feed[is_future].assign(_split="future")
feed_rows = pd.concat([past, future], ignore_index=True)
feed_rows = pd.DataFrame({
    "URL": feed_rows["url"], "label": 0,  # raw PhiUSIIL convention: 0 = phishing
    "_is_aug": 0, "_src": "feed", "_split": feed_rows["_split"],
})
print(f"Feed phishing: {len(feed):,} harvested, {len(held_out):,} held-out groups -> "
      + ", ".join(f"{k}={v:,}" for k, v in feed_rows["_split"].value_counts().items()))
df = pd.concat([df[~df["URL"].isin(set(feed_rows["URL"]))], feed_rows], ignore_index=True)
df = df.drop_duplicates(subset=["URL"]).reset_index(drop=True)


Loaded 235,795 unique rows from ../PhiUSIIL_Phishing_URL_Dataset.csv
Augmented with 1,173 curated legitimate URLs (235,795 -> 236,543 total rows)


Harvested deep links: 23,504 URLs (incl. 7,649 http/query variants) from 1,370 domains -> train=16,521, test=3,492, val=3,491


Feed phishing: 1,546,734 harvested, 1,414 held-out groups -> train=60,000, val=10,000, future=1,103


In [6]:
y = 1 - df["label"].astype(int)
print(f"Class balance after flip -> Legit(0): {int((y==0).sum()):,} | "
      f"Phish(1): {int((y==1).sum()):,}")

Class balance after flip -> Legit(0): 159,527 | Phish(1): 171,380


In [7]:
print("\nBuilding TLD lookup...")
is_train = df["_split"] == "train"
tld_lookup = (
    df[is_train].assign(_legit=(y[is_train] == 0).astype(int))
      .groupby(df.loc[is_train, "URL"].apply(
          lambda u: u.split("//")[-1].split("/")[0].split(":")[0].rsplit(".", 1)[-1].lower()
          if "." in u.split("//")[-1].split("/")[0] else ""
      ))["_legit"]
      .mean()
      .to_dict()
)
print(f"  {len(tld_lookup)} distinct TLDs")

# Character -> P(char | legit URL). Built from legitimate URLs only, with
# +1 smoothing so any character seen at inference has a non-zero prob.
#
# IMPORTANT: PhiUSIIL recorded 100% of legitimate URLs as "www.<domain>"
# (only ~41% of phishing URLs have www.), so counting raw URL characters
# inflates P('w') / P('.') and makes any bare-domain URL (chatgpt.com,
# github.com, ...) look less "legit-shaped" than it should. Canonicalize
# (strip www.) before counting so the table reflects real character
# distribution, not this labeling artifact -- same helper compute_features()
# uses at inference, so train/serve stay consistent.
print("\nBuilding character-frequency lookup from legit URLs...")
counter = {c: 1 for c in string.printable}
total = sum(counter.values())
legit_urls = df.loc[(y == 0) & is_train, "URL"].astype(str)
for u in legit_urls.sample(min(50_000, len(legit_urls)), random_state=RANDOM_STATE):
    for c in canonicalize_url(u).lower():
        counter[c] = counter.get(c, 0) + 1
        total += 1
char_lookup = {c: v / total for c, v in counter.items()}
print(f"  {len(char_lookup)} distinct characters, total mass = "
      f"{sum(char_lookup.values()):.4f}")

# Save the lookups so features.py can load them at inference
joblib.dump(tld_lookup, os.path.join(OUT, "tld_probability_lookup.joblib"))
joblib.dump(char_lookup, os.path.join(OUT, "char_probability_lookup.joblib"))


Building TLD lookup...


  871 distinct TLDs

Building character-frequency lookup from legit URLs...


  189 distinct characters, total mass = 1.0000


['../outputs\\char_probability_lookup.joblib']

In [8]:
print(f"\nExtracting features from {len(df):,} URLs using compute_features()...")
feats = [compute_features(u, tld_lookup, char_lookup) for u in df["URL"].astype(str)]
X = pd.DataFrame(feats)[URL_ONLY_FEATURES].astype(float)
print(f"  Feature matrix shape: {X.shape}")


Extracting features from 330,907 URLs using compute_features()...


  Feature matrix shape: (330907, 22)


In [9]:
AUG_WEIGHT = 10     # up-weight the small curated legit augmentation set
HARVEST_WEIGHT = 2  # harvested real deep links (much larger set)
# Recent feed phishing; env override only for the validation grid search.
FEED_WEIGHT = float(os.environ.get("FEED_WEIGHT", 1.0))

split = df["_split"].values
src = df["_src"].values
tr, va, te = split == "train", split == "val", split == "test"
X_train, y_train = X[tr], y[tr]
X_val, y_val, src_val = X[va], y[va], src[va]
X_test_all, y_test_all, src_test = X[te], y[te], src[te]
# Headline metrics use the PhiUSIIL(+curated) test split, comparable with
# earlier runs; harvested test pages are reported separately as the
# deep-link false-positive rate.
X_test, y_test = X_test_all[src_test != "harvest"], y_test_all[src_test != "harvest"]
X_test_deep = X_test_all[src_test == "harvest"]
# 2026 feed phishing, first seen AFTER every training URL (time-based test)
X_future = X[split == "future"]

sample_weight_train = np.select(
    [src[tr] == "aug", src[tr] == "harvest", src[tr] == "feed"],
    [AUG_WEIGHT, HARVEST_WEIGHT, FEED_WEIGHT], 1.0,
)
print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test),
      "| Deep-link test:", len(X_test_deep), "| Future feed test:", len(X_future))
print("  harvested rows in train:", int((src[tr] == "harvest").sum()),
      f"(weight x{HARVEST_WEIGHT}) | curated aug rows:", int((src[tr] == "aug").sum()),
      f"(weight x{AUG_WEIGHT}) | feed rows:", int((src[tr] == "feed").sum()),
      f"(weight x{FEED_WEIGHT})")


def score(model, name):
    """Fit (up-weighting augmented rows), evaluate on held-out test set."""
    model.fit(X_train, y_train, sample_weight=sample_weight_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    m = {
        "name": name,
        "accuracy": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, pos_label=1)),
        "recall": float(recall_score(y_test, pred, pos_label=1)),
        "f1_score": float(f1_score(y_test, pred, pos_label=1)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "n_features": len(URL_ONLY_FEATURES),
        # Share of REAL legitimate deep links (unseen domains) wrongly
        # flagged at the default 0.5 threshold.
        "deeplink_fp_rate": float((model.predict(X_test_deep) == 1).mean()),
        # Share of 2026 feed phishing (unseen, later in time) flagged at 0.5.
        "future_feed_recall": float((model.predict(X_future) == 1).mean()),
    }
    print(name)
    print(classification_report(
        y_test, pred, target_names=["Legitimate(0)", "Phishing(1)"], digits=4,
    ))
    return model, m



Train: 237212 | Val: 41843 | Test: 47257 | Deep-link test: 3492 | Future feed test: 1103


  harvested rows in train: 16521 (weight x2) | curated aug rows: 788 (weight x10) | feed rows: 60000 (weight x1.0)


In [10]:
dt = DecisionTreeClassifier(
    max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE,
)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE,
)

dt_model, dt_metrics = score(dt, "DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)")
rf_model, rf_metrics = score(rf, "RandomForestClassifier(n_estimators=200, max_depth=20)")

# Persist both so the Admin Model Analytics page can compare them
joblib.dump(dt_model, os.path.join(OUT, "dt_model.joblib"))
joblib.dump(rf_model, os.path.join(OUT, "rf_model.joblib"))

# Winner (by F1 on the phishing class) becomes the deployment model
winner_name, winner_model = (
    ("RandomForest", rf_model)
    if rf_metrics["f1_score"] >= dt_metrics["f1_score"]
    else ("DecisionTree", dt_model)
)
print(f"\n>>> Winner (higher phishing-class F1): {winner_name}")

joblib.dump(winner_model, os.path.join(OUT, "deployment_model.joblib"))
joblib.dump(URL_ONLY_FEATURES, os.path.join(OUT, "deployment_features.joblib"))

DecisionTreeClassifier(max_depth=12, min_samples_leaf=5)
               precision    recall  f1-score   support

Legitimate(0)     0.9591    0.9892    0.9739     27205
  Phishing(1)     0.9847    0.9428    0.9633     20052

     accuracy                         0.9695     47257
    macro avg     0.9719    0.9660    0.9686     47257
 weighted avg     0.9700    0.9695    0.9694     47257



RandomForestClassifier(n_estimators=200, max_depth=20)
               precision    recall  f1-score   support

Legitimate(0)     0.9742    0.9911    0.9826     27205
  Phishing(1)     0.9876    0.9644    0.9759     20052

     accuracy                         0.9797     47257
    macro avg     0.9809    0.9777    0.9792     47257
 weighted avg     0.9799    0.9797    0.9797     47257


>>> Winner (higher phishing-class F1): RandomForest


['../outputs\\deployment_features.joblib']

In [11]:
# ---------------------------------------------------------------------------
# Fix 5 - decision-threshold tuning on the VALIDATION split (never test).
#   phishing  : smallest threshold whose false-positive rate is <= 1% on
#               PhiUSIIL-val legit AND <= 2% on harvested deep-link val legit
#               (so a hard "phishing" verdict is rarely a false accusation).
#   uncertain : smallest threshold >= 0.5 whose share of harvested deep-link
#               val legit pages flagged is <= 7% (the pre-feed-training
#               model's level at 0.5, 6.9%). Adding recent feed phishing
#               made the model score more legit deep links above 0.5, so a
#               fixed 0.5 doubled "Suspicious" warnings on real pages.
#               Scores between this and the phishing threshold are shown as
#               "Suspicious" (warning, not a block).
# ---------------------------------------------------------------------------
p_val = winner_model.predict_proba(X_val)[:, 1]
legit_val = y_val.values == 0
phi_legit = legit_val & (src_val != "harvest")
feed_val = src_val == "feed"
deep_legit = legit_val & (src_val == "harvest")
phish_val = y_val.values == 1

phish_threshold = 0.5
for t in np.arange(0.50, 0.951, 0.01):
    fpr_phi = (p_val[phi_legit] >= t).mean()
    fpr_deep = (p_val[deep_legit] >= t).mean() if deep_legit.any() else 0.0
    if fpr_phi <= 0.01 and fpr_deep <= 0.02:
        phish_threshold = round(float(t), 2)
        break
uncertain_threshold = 0.5
for t in np.arange(0.50, phish_threshold + 1e-9, 0.01):
    if (p_val[deep_legit] >= t).mean() <= 0.07:
        uncertain_threshold = round(float(t), 2)
        break


def _rates(p, t):
    return {
        "threshold": t,
        "val_recall": float((p[phish_val] >= t).mean()),
        "val_fpr_phiusiil": float((p[phi_legit] >= t).mean()),
        "val_fpr_deeplinks": float((p[deep_legit] >= t).mean()),
        "val_recall_feed_nov_dec_2025": float((p[feed_val] >= t).mean()),
    }


print("Validation @0.50:", _rates(p_val, 0.5))
print(f"Validation @{uncertain_threshold:.2f} (uncertain):", _rates(p_val, uncertain_threshold))
print(f"Validation @{phish_threshold:.2f} (phishing):", _rates(p_val, phish_threshold))
p_test = winner_model.predict_proba(X_test)[:, 1]
p_deep = winner_model.predict_proba(X_test_deep)[:, 1]
p_future = winner_model.predict_proba(X_future)[:, 1]
threshold_report = {
    "phishing": phish_threshold,
    "uncertain": uncertain_threshold,
    "test_phishing_recall_at_threshold": float((p_test[y_test.values == 1] >= phish_threshold).mean()),
    "test_flagged_recall_at_uncertain": float((p_test[y_test.values == 1] >= uncertain_threshold).mean()),
    "test_fpr_at_threshold": float((p_test[y_test.values == 0] >= phish_threshold).mean()),
    "test_deeplink_fpr_at_threshold": float((p_deep >= phish_threshold).mean()),
    "test_deeplink_flagged_at_uncertain": float((p_deep >= uncertain_threshold).mean()),
    "future_feed_recall_at_threshold": float((p_future >= phish_threshold).mean()),
    "future_feed_flagged_at_uncertain": float((p_future >= uncertain_threshold).mean()),
    "feed_weight": FEED_WEIGHT,
}
print("Test with tuned thresholds:", json.dumps(threshold_report, indent=2))

metrics_blob = {
    "winner": winner_name,
    "label_convention": {"0": "Legitimate", "1": "Phishing"},
    "positive_class": "Phishing",
    "decision_tree": dt_metrics,
    "random_forest": rf_metrics,
    "thresholds": threshold_report,
}
joblib.dump(metrics_blob, os.path.join(OUT, "deployment_metrics.joblib"))
with open(os.path.join(OUT, "evaluation_metrics.txt"), "w") as fh:
    fh.write(json.dumps(metrics_blob, indent=2))

bg = X_train.sample(1000, random_state=RANDOM_STATE).to_numpy()
np.save(os.path.join(OUT, "lime_background.npy"), bg)

Validation @0.50: {'threshold': 0.5, 'val_recall': 0.9580552907530981, 'val_fpr_phiusiil': 0.009618329963854684, 'val_fpr_deeplinks': 0.14981380693211113, 'val_recall_feed_nov_dec_2025': 0.9459}
Validation @0.73 (uncertain): {'threshold': 0.73, 'val_recall': 0.9277770211993281, 'val_fpr_phiusiil': 0.004349690620596704, 'val_fpr_deeplinks': 0.06760240618733887, 'val_recall_feed_nov_dec_2025': 0.9054}
Validation @0.89 (phishing): {'threshold': 0.89, 'val_recall': 0.8743474510872032, 'val_fpr_phiusiil': 0.0012865282117257857, 'val_fpr_deeplinks': 0.019765110283586365, 'val_recall_feed_nov_dec_2025': 0.8297}


Test with tuned thresholds: {
  "phishing": 0.89,
  "uncertain": 0.73,
  "test_phishing_recall_at_threshold": 0.9068920805904648,
  "test_flagged_recall_at_uncertain": 0.9408039098344305,
  "test_fpr_at_threshold": 0.0014335600073515898,
  "test_deeplink_fpr_at_threshold": 0.019473081328751432,
  "test_deeplink_flagged_at_uncertain": 0.056987399770904924,
  "future_feed_recall_at_threshold": 0.7796917497733454,
  "future_feed_flagged_at_uncertain": 0.8422484134179511,
  "feed_weight": 1.0
}


In [12]:
print("\nSanity check on a few well-known URLs (0 = Legit, 1 = Phish):")
check_urls = [
    "https://chatgpt.com",
    "https://www.google.com/search?q=google+translate",
    "https://github.com/anthropics/anthropic-sdk-python",
    "https://en.wikipedia.org/wiki/Main_Page",
    "http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443",
    "http://192.168.0.1/login.php?redirect=%2Fadmin",
]
for u in check_urls:
    fv = np.array([[compute_features(u, tld_lookup, char_lookup)[n]
                    for n in URL_ONLY_FEATURES]])
    p = winner_model.predict(fv)[0]
    prob = winner_model.predict_proba(fv)[0][1]
    print(f"  P(phish)={prob:.3f}  pred={int(p)}  {u}")

print("\nSaved to ./outputs:")
for f in [
    "deployment_model.joblib", "deployment_features.joblib",
    "deployment_metrics.joblib", "dt_model.joblib", "rf_model.joblib",
    "tld_probability_lookup.joblib", "char_probability_lookup.joblib",
    "lime_background.npy", "evaluation_metrics.txt",
]:
    print(f"  - {f}")



Sanity check on a few well-known URLs (0 = Legit, 1 = Phish):
  P(phish)=0.044  pred=0  https://chatgpt.com
  P(phish)=0.031  pred=0  https://www.google.com/search?q=google+translate


C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\De

  P(phish)=0.017  pred=0  https://github.com/anthropics/anthropic-sdk-python
  P(phish)=0.541  pred=1  https://en.wikipedia.org/wiki/Main_Page
  P(phish)=0.954  pred=1  http://paypa1-secure-login.win/verify.php?id=aGVsbG8=&acct=443


C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\Degree FYP\AI Phishing Detection System\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Inti course folder\De

  P(phish)=0.993  pred=1  http://192.168.0.1/login.php?redirect=%2Fadmin

Saved to ./outputs:
  - deployment_model.joblib
  - deployment_features.joblib
  - deployment_metrics.joblib
  - dt_model.joblib
  - rf_model.joblib
  - tld_probability_lookup.joblib
  - char_probability_lookup.joblib
  - lime_background.npy
  - evaluation_metrics.txt
